# 00 - Oracle, phase kickback và bài toán Deutsch

Đây là bài bản lề giữa "biết dùng cổng" và "hiểu thuật toán". Gần như mọi thuật toán lượng tử
cơ bản (Deutsch-Jozsa, Bernstein-Vazirani, Grover, phase estimation) đều dùng đúng một thủ đoạn:

> Nhét câu trả lời của một hàm vào **phase** của amplitude, rồi dùng $H$ để biến phase thành bit đọc được.

Thủ đoạn đó gọi là **phase kickback**. Bài này xây nó từ đầu rồi dùng ngay vào
bài toán Deutsch.

Cần trước: `00_fundamental/01_cong_1_qubit.ipynb` và `00_fundamental/02_cong_nhieu_qubit.ipynb`.

In [1]:
import numpy as np

from qiskit import ClassicalRegister, QuantumCircuit, QuantumRegister
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram
from qiskit_aer.primitives import SamplerV2
from IPython.display import display

import qiskit

print("qiskit:", qiskit.__version__)

sampler = SamplerV2()


def ket(sv, nd=3):
    """In statevector dưới dạng tổng các ket, bỏ qua amplitude ~ 0."""
    phan = []
    for i, bien_do in enumerate(sv.data):
        if abs(bien_do) < 1e-9:
            continue
        bits = format(i, f"0{sv.num_qubits}b")
        if abs(bien_do.imag) < 1e-9:
            he_so = f"{bien_do.real:+.{nd}f}"
        elif abs(bien_do.real) < 1e-9:
            he_so = f"{bien_do.imag:+.{nd}f}i"
        else:
            he_so = f"({bien_do.real:+.{nd}f}{bien_do.imag:+.{nd}f}i)"
        phan.append(f"{he_so}|{bits}>")
    return "  ".join(phan) if phan else "0"


def do(qc, shots=1024, reg="c"):
    """Chạy mạch và trả về counts từ thanh ghi cổ điển tên `reg`."""
    ket_qua = sampler.run([qc], shots=shots).result()[0]
    return getattr(ket_qua.data, reg).get_counts()

qiskit: 2.5.2


## 1. Oracle: đóng gói một hàm cổ điển thành mạch lượng tử

Cho hàm $f:\{0,1\}^n \to \{0,1\}$. Ta không được xem ruột $f$, chỉ được **gọi** nó - giống
gọi một API. Trong thế giới lượng tử, $f$ phải được đóng gói thành một **cổng unitary**, và
unitary thì bắt buộc phải đảo ngược được.

Vấn đề: $|x\rangle \mapsto |f(x)\rangle$ **không** đảo ngược được (nhiều $x$ cho cùng $f(x)$).
Cách chuẩn để lách: thêm một qubit phụ (**ancilla**) và ghi kết quả bằng phép XOR:

$$U_f\,\lvert x\rangle\lvert y\rangle = \lvert x\rangle\,\lvert y \oplus f(x)\rangle$$

Dạng này luôn đảo ngược được (áp hai lần thì về chỗ cũ, vì $y \oplus f \oplus f = y$). Người ta
gọi nó là **bit-flip oracle**: nó lật ancilla khi $f(x) = 1$.

Với $n = 1$, có đúng 4 hàm khả dĩ:

| $f$ | $f(0)$ | $f(1)$ | Loại | Mạch $U_f$ |
|---|---|---|---|---|
| $f_0$ | 0 | 0 | hằng (constant) | không làm gì |
| $f_1$ | 1 | 1 | hằng | `x(ancilla)` |
| $f_2$ | 0 | 1 | cân bằng (balanced) | `cx(x, ancilla)` |
| $f_3$ | 1 | 0 | cân bằng | `cx(x, ancilla)` + `x(ancilla)` |

In [ ]:
def oracle_deutsch(loai):
    """U_f trên 2 qubit: qubit 0 = x (dữ liệu), qubit 1 = ancilla."""
    qc = QuantumCircuit(2, name=f"U_{loai}")
    if loai == "f0":           # f(x) = 0
        pass
    elif loai == "f1":         # f(x) = 1
        qc.x(1)
    elif loai == "f2":         # f(x) = x
        qc.cx(0, 1)
    elif loai == "f3":         # f(x) = NOT x
        qc.cx(0, 1)
        qc.x(1)
    return qc


def ket_co_so(sv):
    """Với trạng thái là ĐÚNG một ket cơ sở, trả về chuỗi bit của nó."""
    i = int(np.argmax(np.abs(sv.data)))
    return format(i, f"0{sv.num_qubits}b")


# Kiểm tra oracle chạy đúng bảng chân trị: U_f|x>|0> = |x>|f(x)>
print("chuỗi bit đọc là '<ancilla><x>' (ancilla = qubit 1 nên nằm bên trái)\n")
for loai in ["f0", "f1", "f2", "f3"]:
    ra = []
    for x in [0, 1]:
        qc = QuantumCircuit(2)
        if x:
            qc.x(0)
        qc.compose(oracle_deutsch(loai), inplace=True)
        bits = ket_co_so(Statevector(qc))
        ra.append(f"f({x})={bits[0]}")      # ancilla đang giữ f(x)
    print(f"  {loai}: {', '.join(ra)}")

## 2. Phase kickback: đặt ancilla ở $\lvert -\rangle$ thay vì $\lvert 0\rangle$

Đây là bước then chốt. Nhớ rằng $X\lvert -\rangle = -\lvert -\rangle$: $\lvert -\rangle$ là
**vector riêng** của $X$ với trị riêng $-1$. Mà bit-flip oracle chính là "áp $X$ lên ancilla
khi $f(x)=1$". Nên nếu ancilla đang ở $\lvert -\rangle$:

$$U_f\,\lvert x\rangle\lvert -\rangle = (-1)^{f(x)}\,\lvert x\rangle\lvert -\rangle$$

Ancilla **không đổi** - nó chỉ "đập" lại một dấu $(-1)^{f(x)}$ vào qubit dữ liệu. Câu trả lời
của $f$ đã chuyển từ *bit của ancilla* sang *phase của qubit dữ liệu*. Đó là "kickback".

Điều kỳ diệu nằm ở bước tiếp theo: cho $x$ ở trạng thái superposition $\lvert +\rangle$ thì một
lần gọi oracle đã đập dấu lên **cả hai** giá trị $x$ cùng lúc:

$$U_f\,\lvert +\rangle\lvert -\rangle
= \tfrac{1}{\sqrt2}\big((-1)^{f(0)}\lvert 0\rangle + (-1)^{f(1)}\lvert 1\rangle\big)\lvert -\rangle$$

In [ ]:
def phan_du_lieu(sv, n):
    """Tách riêng n qubit dữ liệu khi ancilla (qubit cao nhất) đang ở |->.

    Trạng thái lúc đó có dạng |psi_du_lieu> (x) |->, nên khối 2^n amplitude đầu tiên
    (khối ứng với ancilla = 0) chính là psi_du_lieu chia cho sqrt(2).
    """
    return Statevector(sv.data[: 2 ** n] * np.sqrt(2))


plus = Statevector([1 / np.sqrt(2), 1 / np.sqrt(2)])
minus = Statevector([1 / np.sqrt(2), -1 / np.sqrt(2)])

print("Ancilla ở |->: oracle chỉ đập dấu lên qubit dữ liệu, ancilla giữ nguyên")
print("(chuỗi |ab> với a = ancilla = qubit 1, b = dữ liệu = qubit 0)\n")

for loai in ["f0", "f1", "f2", "f3"]:
    qc = QuantumCircuit(2)
    qc.x(1)
    qc.h(1)           # ancilla -> |->
    qc.h(0)           # dữ liệu  -> |+>
    qc.compose(oracle_deutsch(loai), inplace=True)
    sv = Statevector(qc)
    du_lieu = phan_du_lieu(sv, 1)

    ten = "?"
    for nhan, muc_tieu in [("|+>", plus), ("|->", minus)]:
        if np.allclose(du_lieu.data, muc_tieu.data) or np.allclose(du_lieu.data, -muc_tieu.data):
            ten = nhan

    print(f"  {loai}: cả hai qubit        = {ket(sv)}")
    print(f"       riêng qubit dữ liệu = {ket(du_lieu)}  ->  {ten}\n")

Đọc kết quả trên cho kỹ: dòng in cả hai qubit có dấu bị trộn lẫn với dấu của ancilla
$\lvert -\rangle$, nên muốn so sánh thì phải xem dòng thứ hai - dòng đã tách riêng qubit dữ liệu.

Kết luận: với $f_0$, $f_1$ (hàm hằng) hai amplitude của qubit dữ liệu **cùng dấu**, còn với
$f_2$, $f_3$ (cân bằng) thì **trái dấu**. Nói cách khác, qubit dữ liệu đang ở
$\lvert +\rangle$ hoặc $\lvert -\rangle$ (sai khác dấu tổng thể):

$$f \text{ hằng} \Rightarrow \pm\lvert +\rangle, \qquad
  f \text{ cân bằng} \Rightarrow \pm\lvert -\rangle$$

Và bài `00_fundamental/00` đã dạy cách phân biệt hai trạng thái này: **cho qua $H$ rồi đo**.
$H\lvert +\rangle = \lvert 0\rangle$, $H\lvert -\rangle = \lvert 1\rangle$.

## 3. Bài toán Deutsch

> Cho $f:\{0,1\}\to\{0,1\}$ dưới dạng oracle. Hỏi $f$ **hằng** hay **cân bằng**?

Cổ điển: phải gọi oracle **2 lần** (tính $f(0)$ và $f(1)$ rồi so sánh). Không có cách nào ít hơn,
vì biết $f(0)$ một mình không nói lên điều gì.

Lượng tử: **1 lần gọi**. Mạch gọn lỏn:

```
        ┌───┐     ┌───────┐     ┌───┐┌─┐
x:   ───┤ H ├─────┤       ├─────┤ H ├┤M├     -> 0 = hằng, 1 = cân bằng
        └───┘     │  U_f  │     └───┘└╥┘
        ┌───┐┌───┐│       │           ║
anc: ───┤ X ├┤ H ├┤       ├───────────╫──    (không cần đo)
        └───┘└───┘└───────┘           ║
```

Kết quả đo được chính là $f(0) \oplus f(1)$: bằng 0 khi hai giá trị giống nhau (hằng),
bằng 1 khi khác nhau (cân bằng).

In [ ]:
def deutsch(loai):
    qc = QuantumCircuit(2, 1)       # 2 qubit, 1 bit cổ điển (chỉ đo qubit dữ liệu)
    qc.x(1)
    qc.h(1)                         # ancilla -> |->
    qc.h(0)                         # dữ liệu  -> |+>
    qc.barrier()
    qc.compose(oracle_deutsch(loai), inplace=True)
    qc.barrier()
    qc.h(0)
    qc.measure(0, 0)
    return qc


print(deutsch("f2").draw())

print("\nKết quả (1 lần gọi oracle mỗi hàm):")
for loai, that in [("f0", "hằng"), ("f1", "hằng"), ("f2", "cân bằng"), ("f3", "cân bằng")]:
    c = do(deutsch(loai), shots=1024)
    doc_duoc = "hằng" if list(c) == ["0"] else "cân bằng"
    print(f"  {loai}: counts={c}  -> kết luận: {doc_duoc:9s} (đáp án: {that})")

Lưu ý hai chi tiết kỹ thuật dễ sai:

- Mạch khai báo `QuantumCircuit(2, 1)` và chỉ `measure(0, 0)`. Nếu dùng `measure_all()` thì
  ancilla cũng bị đo, và vì ancilla đang ở $\lvert -\rangle$ nên nó cho bit **ngẫu nhiên** -
  chuỗi counts sẽ có một bit rác làm bạn tưởng thuật toán sai.
- `barrier()` chỉ để bản vẽ cho dễ đọc, không ảnh hưởng kết quả.

## 4. Deutsch-Jozsa: mở rộng ra $n$ bit

> Cho $f:\{0,1\}^n\to\{0,1\}$, **biết trước** rằng $f$ hoặc hằng, hoặc cân bằng
> (đúng một nửa đầu vào cho 0, một nửa cho 1). Hỏi $f$ thuộc loại nào?

Cổ điển, trường hợp xấu nhất phải gọi $2^{n-1}+1$ lần: kiểm $2^{n-1}$ đầu vào đầu tiên mà
kết quả vẫn giống nhau thì chưa kết luận được gì, phải gọi thêm một lần nữa.

Lượng tử: vẫn **1 lần gọi**. Mạch y hệt Deutsch, chỉ là cho $n$ qubit dữ liệu:

1. $H$ lên tất cả $n$ qubit dữ liệu, ancilla về $\lvert -\rangle$
2. gọi oracle một lần
3. $H$ lên tất cả $n$ qubit dữ liệu
4. đo $n$ qubit đó: ra **toàn 0** thì $f$ hằng, ra **bất kỳ thứ gì khác** thì $f$ cân bằng

Oracle cân bằng dùng ở đây: $f(x) = x_0 \oplus x_1 \oplus \dots \oplus x_{n-1}$ (XOR tất cả các bit),
dựng bằng `cx` từ mỗi qubit dữ liệu vào ancilla.

In [ ]:
def deutsch_jozsa(n, loai):
    qc = QuantumCircuit(n + 1, n)
    qc.x(n)
    qc.h(n)                  # ancilla -> |->
    qc.h(range(n))
    qc.barrier()
    if loai == "cân bằng":   # f(x) = XOR các bit của x
        for i in range(n):
            qc.cx(i, n)
    # loai == "hằng": oracle không làm gì (f(x) = 0)
    qc.barrier()
    qc.h(range(n))
    qc.measure(range(n), range(n))
    return qc


print(deutsch_jozsa(3, "cân bằng").draw())

for loai in ["hằng", "cân bằng"]:
    c = do(deutsch_jozsa(3, loai), shots=1024)
    ket_luan = "hằng" if list(c) == ["000"] else "cân bằng"
    print(f"\n{loai:9s}: counts={c} -> kết luận: {ket_luan}")

In [ ]:
# So sánh số lần gọi oracle: cổ điển (xấu nhất) vs lượng tử
print(f"{'n':>3} {'cổ điển (xấu nhất)':>20} {'lượng tử':>10}")
for n in [1, 2, 3, 10, 20, 50]:
    print(f"{n:>3} {2 ** (n - 1) + 1:>20,} {1:>10}")

## Tóm tắt

| Ý tưởng | Nội dung |
|---|---|
| Bit-flip oracle | $U_f\lvert x\rangle\lvert y\rangle = \lvert x\rangle\lvert y\oplus f(x)\rangle$, luôn đảo ngược được |
| Phase kickback | ancilla ở $\lvert -\rangle$ thì $U_f\lvert x\rangle\lvert -\rangle = (-1)^{f(x)}\lvert x\rangle\lvert -\rangle$ |
| Vì sao dùng $\lvert -\rangle$ | $\lvert -\rangle$ là vector riêng của $X$ với trị riêng $-1$ |
| Đọc phase ra bit | $H$ biến $\pm$ thành $0/1$: $H\lvert +\rangle=\lvert 0\rangle$, $H\lvert -\rangle=\lvert 1\rangle$ |
| Deutsch | 1 lần gọi thay vì 2 |
| Deutsch-Jozsa | 1 lần gọi thay vì $2^{n-1}+1$ |

Điểm quan trọng về bản chất: thuật toán **không** tính $f$ nhanh hơn. Nó chỉ trả lời được một
câu hỏi **toàn cục** về $f$ (hằng hay cân bằng) mà không cần biết từng giá trị $f(x)$ - đó là
kiểu lợi thế mà hầu hết thuật toán lượng tử khai thác.

### Thử tự làm

1. Viết oracle cân bằng **khác**, ví dụ $f(x) = x_0$ (chỉ phụ thuộc bit thấp nhất), và kiểm tra
   Deutsch-Jozsa vẫn kết luận đúng.
2. Thử đặt ancilla ở $\lvert 0\rangle$ thay vì $\lvert -\rangle$ trong mạch Deutsch. Giải thích
   vì sao kết quả trở thành vô nghĩa.
3. Bỏ cổng $H$ cuối cùng đi rồi đo - bạn đọc được gì, và vì sao nó không giúp ích?

Bài tiếp theo: `01_bernstein_vazirani.ipynb`.